# Ghosting — entrenamiento en GPU

Entrena el imputador residual sobre GPU gratuita (Kaggle T4/P100 o Colab T4).

**Antes de empezar**, en tu laptop:
```bash
python scripts/06_export_bundle.py
```
Eso genera `ghosting_bundle.zip` (~36 MB con los 9 partidos reales: datos ya
procesados a 5 fps + 45 min, más el código completo).

**No se descarga nada de HuggingFace aquí.** Bajar Sportec dentro de la sesión
gastaría 15–30 minutos de GPU alquilada en red y parseo de XML que ya hiciste
en casa.

### Kaggle (recomendado)
Sube el zip como *Dataset* privado → queda persistente entre sesiones, no hay
que resubirlo. 30 h/semana de GPU, sesiones de hasta 9 h.

### Colab
Sube el zip a Drive y monta. La sesión libre se puede reclamar sin aviso, así
que **guarda el checkpoint en Drive**, no en el disco local del contenedor.

In [ ]:
# --- 1. Comprobar GPU ---
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True,
                     text=True).stdout.strip() or 'sin nvidia-smi')
print('CUDA disponible:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Activa la GPU: Entorno > Acelerador > GPU'

In [ ]:
# --- 2. Desempaquetar ---
import zipfile, os, glob, sys

# Kaggle: el dataset se monta en /kaggle/input/<nombre>/
# Colab:  monta Drive y apunta la ruta al zip
cands = glob.glob('/kaggle/input/**/ghosting_bundle.zip', recursive=True)
if not cands:
    from google.colab import drive
    drive.mount('/content/drive')
    cands = glob.glob('/content/drive/MyDrive/**/ghosting_bundle.zip',
                      recursive=True)
assert cands, 'No encuentro ghosting_bundle.zip'

os.makedirs('/content/gh', exist_ok=True)
with zipfile.ZipFile(cands[0]) as z:
    z.extractall('/content/gh')
os.chdir('/content/gh')
sys.path.insert(0, '/content/gh/src')
print('partidos:', sorted(os.listdir('data/processed')))

In [ ]:
# --- 3. Dependencias (torch ya viene instalado) ---
!pip install -q kloppy tqdm 2>/dev/null; echo ok

In [ ]:
# --- 4. Estimar el coste ANTES de comprometerse ---
# Mide el ritmo real y extrapola. Nunca lances una corrida larga sin esto.
!python scripts/04_train.py --long --epochs 100 --dry-run 60 --device cuda

In [ ]:
# --- 5. Entrenar ---
# El lote sube mucho respecto a CPU: en GPU la memoria sobra y el paralelismo
# se aprovecha. --monitor '>9.6s' vigila el régimen que es objetivo del
# experimento, no la mediana global.
!python scripts/04_train.py --long --epochs 100 --batch 64 \
    --monitor '>9.6s' --patience 12 --device cuda --no-bar

In [ ]:
# --- 6. Evaluar contra B4 en el test interno ---
!python scripts/05_evaluate_model.py --boot 1000

In [ ]:
# --- 7. Test externo congelado (Metrica) ---
# Estos dos partidos no se han tocado en ningún momento del desarrollo.
!python scripts/05_evaluate_model.py --matches metrica_1 metrica_2 --boot 1000

In [ ]:
# --- 8. Bajar resultados ---
import shutil, glob
shutil.make_archive('/content/resultados', 'zip', 'reports')
try:
    from google.colab import files; files.download('/content/resultados.zip')
except ImportError:
    shutil.copy('/content/resultados.zip', '/kaggle/working/')
    print('-> /kaggle/working/resultados.zip')